<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Test changes online

**[Test changes online](https://learning.nextia-ai.com/courses/search/m08/test-changes-online/)** · Search and Ranking: Find the Best Match · Module 8, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** simulate team-draft interleaving between two designs with a click model, test the result with a sign test, and see how looking too often finds winners that do not exist.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 40 MB). No account, no key and no model download. |
| **You should know** | NDCG@10 and confidence intervals, from [Build a test set that represents your users](https://learning.nextia-ai.com/courses/search/m02/build-a-test-set-that-represents-your-users/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m08/test-changes-online/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M08-L02-test-changes-online/test-changes-online-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 41 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/embeddings/products.npy": "f422870c4ab75364fad883fa8858bd7f31f2591bad07753a8f713a0872824ab0",
    "data/embeddings/queries.npy": "5333da6bd9ac44ec1bf92ddf916ec60cb557466b90b5075711f59c8e61f96253",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two designs and their top 10

You compare two designs on Larkfield's test queries in English, in French (machine-translated) and with planted misspellings: BM25 on the title and description (A), and a hybrid of BM25 and dense search by reciprocal rank fusion (B). This is a simple version of the project's runs. Run the cell: it builds both top-10 lists for every query and prints NDCG@10 (the offline test).

In [ ]:
import re, bm25s, compare
cat = catalogue()
q = queries()
j = judgments()
ids = cat["product_id"].to_numpy()
tok = lambda t: [w for w in re.split(r"[^\w]+", str(t).lower()) if w]
r = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
r.index([tok(t + " " + d) for t, d in zip(cat["title"], cat["description"])], show_progress=False)
P = np.load("data/embeddings/products.npy").astype(np.float32)
Q = np.load("data/embeddings/queries.npy").astype(np.float32)

def bm25_top(text, k=100):
    words = [w for w in tok(text) if w in r.vocab_dict]
    if not words:
        return []
    s = r.get_scores(words)
    top = np.argsort(-s, kind="stable")[:k]
    return [ids[i] for i in top if s[i] > 0]

def dense_top(row, k=100):
    s = P @ Q[row]
    return [ids[i] for i in np.argsort(-s, kind="stable")[:k]]

def rrf(lists, k=60):
    score = {}
    for lst in lists:
        for rank, pid in enumerate(lst, 1):
            score[pid] = score.get(pid, 0) + 1 / (k + rank)
    return sorted(score, key=lambda p: -score[p])

test = q[(q["split"] == "test") & q["source"].isin(["wands", "translated", "typo"])]
lists = {}
for row, qid, text in zip(test.index, test["query_id"], test["query"]):
    a = bm25_top(text)
    lists[qid] = (a[:10], rrf([a, dense_top(row)])[:10])
for name, i in (("bm25", 0), ("hybrid", 1)):
    print(name, round(np.nanmean([compare.ndcg(v[i], j.get(qid, {})) for qid, v in lists.items()]), 3))

> **Check.** BM25 0.468, hybrid 0.532. Offline, the hybrid is better. Would people prefer it too? Nobody can click in a notebook, so a **click model** clicks for them: the same rules that made the course's synthetic log.

## 2. The click model

A person looks at position k with the chance 1/k. If they look, they click an exact product with the chance 0.65, a partial one with 0.25 and an irrelevant one with 0.05. These numbers are **assumptions** that we chose, not measurements of real people.

In [ ]:
CLICK = {2: 0.65, 1: 0.25, 0: 0.05}     # click if looked at: exact, partial, irrelevant (assumptions)

def examine(k):
    return 1 / k                          # the chance that a person looks at position k

def simulate_clicks(shown, judged, rng):
    """One person and one page: a 0 or 1 for each position."""
    return [int(rng.random() < examine(k) * CLICK[judged.get(pid, 0)]) for k, pid in enumerate(shown, 1)]

rng = np.random.default_rng(0)
qid = test["query_id"].iloc[0]
print(test["query"].iloc[0], simulate_clicks(lists[qid][0], j[qid], rng))

## 3. Team-draft interleaving

**Interleaving** shows one page that mixes both lists. In **team-draft** interleaving, A and B pick in turns, like two captains choosing players. A coin decides who picks first in each round. Each picks its best product that is not on the page yet. The cell shows a tiny example with four products.

In [ ]:
def team_draft(a, b, rng, n=10):
    """A and B take turns (a coin decides who picks first in each round) to add their best product not shown yet."""
    out, team, ia, ib = [], [], 0, 0
    while len(out) < n and (ia < len(a) or ib < len(b)):
        for t in (("A", "B") if rng.random() < 0.5 else ("B", "A")):
            src, i = (a, ia) if t == "A" else (b, ib)
            while i < len(src) and src[i] in out:
                i += 1
            if i < len(src) and len(out) < n:
                out.append(src[i]); team.append(t); i += 1
            if t == "A":
                ia = i
            else:
                ib = i
    return out, team

a = ["P1", "P2", "P3", "P4"]
b = ["P3", "P1", "P5", "P6"]
shown, team = team_draft(a, b, np.random.default_rng(1), n=4)
pd.DataFrame({"position": range(1, 5), "product": shown, "team": team})

> **Read the table.** Each product remembers its team. If a person clicks more of A's products than B's, A wins this search. A tie gives no winner.

## 4. Your turn: who won this search?

Complete `winner`: count the clicks on each team's products.

In [ ]:
def winner(clicks, team):
    ca = sum(c for c, t in zip(clicks, team) if t == "A")
    cb = sum(c for c, t in zip(clicks, team) if t == "B")
    return "A" if ca > cb else "B" if cb > ca else None

w_example = winner([0, 1, 1, 0], ["A", "B", "B", "A"])
print(w_example, winner([1, 0, 1, 0], ["B", "A", "B", "A"]))

In [ ]:
expect('the winner of the first example', w_example, 'B')

## 5. Run the test

Now simulate 5,000 searches. Each search picks a random test query, builds the interleaved page, and lets the click model click. After every 100 searches, a **sign test** asks: could A and B win equally often, and this gap is only luck? Its answer is a **p-value**. Below 0.05, we call the result clear.

In [ ]:
import math

def sign_test_p(wins_a, wins_b):
    """Two-sided p-value that A and B win equally often."""
    n = wins_a + wins_b
    if n == 0:
        return 1.0
    return math.erfc(max((abs(wins_a - wins_b) - 1) / math.sqrt(n), 0) / math.sqrt(2))

def interleave(lists, seed, searches=5000, every=100):
    """Wins of A and B, and the p-value after every `every` searches."""
    rng = np.random.default_rng(seed)
    qids = list(lists)
    wa = wb = 0
    trace = []
    for s in range(1, searches + 1):
        qid = qids[rng.integers(len(qids))]
        shown, team = team_draft(*lists[qid], rng)
        w = winner(simulate_clicks(shown, j.get(qid, {}), rng), team)
        wa += w == "A"; wb += w == "B"
        if s % every == 0:
            trace.append(sign_test_p(wa, wb))
    return wa, wb, trace

wa, wb, trace = interleave(lists, seed=0)
first = next((100 * (i + 1) for i, p in enumerate(trace) if p < 0.05), None)
print(f"BM25 wins {wa}, hybrid wins {wb}; clear (p < 0.05) after {first} searches")
plt.figure(figsize=(6, 3)); plt.plot(range(100, 5001, 100), trace); plt.axhline(0.05, color="red", ls="--")
plt.yscale("log"); plt.xlabel("searches"); plt.ylabel("p-value"); plt.title("BM25 against hybrid (simulated clicks)")
plt.grid(alpha=0.3); plt.show()

> **Check.** With these seeds, the hybrid wins 1269 searches and BM25 927, and the test is clear after 500 searches. The offline difference is large, so few searches are enough.

## 6. Your turn: a test with no real difference

Now compare the hybrid **with itself**. There is no real difference, so every "winner" is a false alarm. Count how many of 20 tests find a winner when you look after every 100 searches.

In [ ]:
same = {qid: (v[1], v[1]) for qid, v in lists.items()}   # hybrid against itself: no real difference
traces = [interleave(same, seed, searches=5000)[2] for seed in range(20)]   # 20 tests, a p-value every 100
false_winners = sum(any(p < 0.05 for p in tr) for tr in traces)   # someone who looks every 100 searches
print(false_winners, "of 20 tests found a winner")

In [ ]:
expect('false_winners', false_winners, 4)

> **Check.** 4 of 20 tests found a winner that does not exist. Each look is one more chance for luck to cross the line. This is **peeking**. Now look only once, at the number of searches you planned:

In [ ]:
once = sum(tr[-1] < 0.05 for tr in traces)   # look only once, at the planned 5,000 searches
print(once, "of 20 tests found a winner when we looked once")

> **Check.** 0 of 20. Decide the number of searches before you start, and look once. In the project: `search interleave hybrid-rrf ltr --seeds 5`, and `search interleave hybrid-rrf hybrid-rrf --seeds 20` for the test with no real difference.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Keep the index fresh](https://learning.nextia-ai.com/courses/search/m08/keep-the-index-fresh/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m08/test-changes-online/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.